#### 1. Install dependencies

In [63]:
%pip install --quiet google-adk requests

#### 2. Imports/configuration

Enter your Google Maps Geocoding API key after running cell below. Re-enter it after the runtime restarts.

In [45]:
import getpass
from typing import Dict, Any

import os

import requests

# --- Configuration ---
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-06373caf63ce"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

# Geocoding API key is masked in the UI and not stored.
GOOGLE_MAPS_API_KEY = getpass.getpass("Geocoding API key: ")

# The NWS API requires a User-Agent header.
NWS_USER_AGENT = "challenge-1-weather-agent-colab (student-02-730f46eb80e1@qwiklabs.net)"

Geocoding API key: ··········


#### 3. Tool: Get weather from the National Weather Service API

Takes latitude and longitude and returns the current forecast.

In [46]:
def get_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Retrieve the current weather forecast for a location.

    Use the U.S. National Weather Service (NWS) API. The NWS API
    resolves the latitude/longitude to a forecast grid endpoint, then
    fetches the forecast periods from that endpoint.

    Args:
        latitude: The latitude of the location in decimal degrees.
        longitude: The longitude of the location in decimal degrees.

    Returns:
        A dictionary containing the forecast. On success it has the keys:
            ``status`` (str): "success".
            ``period`` (str): The name of the forecast period (e.g. "Tonight").
            ``temperature`` (str): The temperature and unit (e.g. "72 F").
            ``forecast`` (str): A short human-readable forecast.
            ``detailed_forecast`` (str): A longer forecast description.
        On failure it returns a dictionary with keys ``status`` ("error")
        and ``error_message`` (str).
    """
    headers = {"User-Agent": NWS_USER_AGENT, "Accept": "application/geo+json"}

    try:
        # Step 1: Resolve the point to a forecast grid endpoint.
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        # Step 2: Fetch the forecast from the resolved endpoint.
        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        if not periods:
            return {
                "status": "error",
                "error_message": "No forecast periods were returned for this location.",
            }

        current = periods[0]
        return {
            "status": "success",
            "period": current["name"],
            "temperature": f"{current['temperature']} {current['temperatureUnit']}",
            "forecast": current["shortForecast"],
            "detailed_forecast": current["detailedForecast"],
        }
    except requests.exceptions.RequestException as exc:
        return {
            "status": "error",
            "error_message": (
                f"Failed to retrieve weather data: {exc}."
            ),
        }
    except (KeyError, IndexError) as exc:
        return {
            "status": "error",
            "error_message": f"Unexpected response format from NWS API: {exc}.",
        }

#### 4. Tool: Geocode a place using the Google Maps Geocoding API

Converts city/state into latitude and longitude.

In [47]:
def geocode_place(place: str) -> Dict[str, Any]:
    """Convert a city/state into latitude and longitude coordinates.

    Uses the Google Maps Geocoding API to resolve a place
    description (i.e. ``"Austin, TX"`` or ``"Seattle, Washington"``)
    into coordinates.

    Args:
        place: A free-form place description such as a city and state.

    Returns:
        A dictionary containing the geocoding result. On success it has
        the keys:
            ``status`` (str): "success".
            ``latitude`` (float): The latitude in decimal degrees.
            ``longitude`` (float): The longitude in decimal degrees.
            ``formatted_address`` (str): The normalized address string.
        On failure it returns a dictionary with keys ``status`` ("error")
        and ``error_message`` (str).
    """
    endpoint = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": GOOGLE_MAPS_API_KEY}

    try:
        resp = requests.get(endpoint, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        if data.get("status") != "OK" or not data.get("results"):
            return {
                "status": "error",
                "error_message": (
                    f"Could not geocode '{place}'. API status: "
                    f"{data.get('status', 'UNKNOWN')}."
                ),
            }

        result = data["results"][0]
        location = result["geometry"]["location"]
        return {
            "status": "success",
            "latitude": location["lat"],
            "longitude": location["lng"],
            "formatted_address": result["formatted_address"],
        }
    except requests.exceptions.RequestException as exc:
        return {
            "status": "error",
            "error_message": f"Failed to reach the Geocoding API: {exc}.",
        }

#### 5. Build the agent

Register the previous two functions as tools.

In [55]:
from google.adk.agents import Agent

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description=(
        "An agent that answers questions about the current weather for any "
        "U.S. city or state."
    ),
    instruction=(
        "You are a helpful weather assistant. When a user asks about the "
        "weather in a place (such as a city and state), follow these steps:\n"
        "1. Call the `geocode_place` tool with the place the user mentioned "
        "to obtain its latitude and longitude.\n"
        "2. Call the `get_weather` tool with that latitude and longitude to "
        "retrieve the forecast.\n"
        "3. Report the weather to the user in a friendly, concise sentence, "
        "including the temperature and a short description.\n\n"
        "If either tool returns an error status, apologize and explain the "
        "problem clearly. Remember that the National Weather Service only "
        "covers locations within the United States, so if a user asks about "
        "a place outside the U.S., let them know you can only provide U.S. "
        "weather. If the user does not specify a state, ask for clarification "
        "before geocoding."
        "4. Infer the state if you can deduce fairly accurately. \n"
        "5. If the user provides a city name that's also a state name, "
        "assume it's a city and deduce which state it's in if possible."
    ),
    tools=[geocode_place, get_weather],
)

#### 6. Run the agent

Start the conversation with the agent.


In [56]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "weather_app"
USER_ID = "user_1"
SESSION_ID = "session_1"

session_service = InMemorySessionService()

runner = Runner(
    agent=weather_agent,
    app_name=APP_NAME,
    session_service=session_service,
)


async def ask_agent(query: str) -> None:
    """Send a query to the weather agent and print the final response.

    Args:
        query: The natural-language question from the user.
    """
    # Ensure a session exists.
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    if session is None:
        await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )

    content = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID, new_message=content
    ):
        if event.is_final_response() and event.content and event.content.parts:
            print("Agent:", event.content.parts[0].text)

#### 7. Interact with the agent

In [59]:
await ask_agent("What's the weather in Dallas, TX?")

Agent: Today, in Dallas, TX, it's 101 F and sunny.


In [60]:
await ask_agent("What's the weather in Portland, Oregon?")

Agent: Today, in Portland, Oregon, it's 93 F and sunny with patchy smoke.


In [62]:
await ask_agent("What's the weather in Seattle, Washington?")

Agent: Today, in Seattle, Washington, it's 83 F and sunny.


In [61]:
await ask_agent("What's the weather like in Juneau?")

Agent: Today, in Juneau, it's 61 F and cloudy.


In [57]:
await ask_agent("What's the weather like in Denver?")

Agent: Today, in Denver, it's 97 F with areas of smoke.


In [58]:
await ask_agent("What about the weather in New York?")

Agent: Today, in New York, it's 87 F and mostly sunny.
